# Modelling V2 — Iterative Improvement

Picks up from `05_modelling.ipynb` (baseline RMSPE: 0.1852).

Each version adds one change, measures the impact, and either keeps or drops it.

**Final result: 0.1140** — 38% improvement over baseline.

In [45]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
import os

warnings.filterwarnings("ignore")

In [31]:
df = pd.read_csv("../data/processed/train_features.csv")

TARGET = "Sales"
FEATURES = df.columns.tolist()
FEATURES.remove("Sales")
FEATURES.remove("Customers")

df = df.sort_values(["Year", "Week"]).reset_index(drop=True)

split_idx = int(len(df) * 0.85)
train_df = df.iloc[:split_idx].copy()
val_df   = df.iloc[split_idx:].copy()

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")

Train: 717,687
Val:   126,651


In [32]:
def rmspe(y_true, y_pred):
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

In [33]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbose": -1
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=500)
]

## V1 — Fix unconverged model (increase num_boost_round to 5000)

Original model stopped at 998 rounds and had not converged.
Early stopping with 5000 rounds finds the true optimum at round 3581.

In [34]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val   = val_df[FEATURES]
y_val   = val_df[TARGET]

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

model_v1 = lgb.train(
    params,
    lgb_train,
    num_boost_round=5000,
    valid_sets=[lgb_val],
    callbacks=callbacks
)

preds_v1 = np.maximum(model_v1.predict(X_val), 0)
rmspe_v1 = rmspe(y_val.values, preds_v1)

print(f"\nV1 RMSPE (5000 rounds): {rmspe_v1:.4f}")
print(f"Best iteration: {model_v1.best_iteration}")


[500]	valid_0's rmse: 1358.66
[1000]	valid_0's rmse: 1229.21
[1500]	valid_0's rmse: 1173.45
[2000]	valid_0's rmse: 1143.92
[2500]	valid_0's rmse: 1126.4
[3000]	valid_0's rmse: 1111.08
[3500]	valid_0's rmse: 1102.62

V1 RMSPE (5000 rounds): 0.1606
Best iteration: 3581


## V2 — Add StoreMeanSales

Store is the top feature — the model spends most trees learning per-store baselines.
Providing that baseline directly as a feature frees trees for finer patterns.

**Rule:** calculate from training data only to prevent leakage.

In [35]:
store_mean = train_df.groupby('Store')['Sales'].mean()

train_df['StoreMeanSales'] = train_df['Store'].map(store_mean)
val_df['StoreMeanSales'] = val_df['Store'].map(store_mean)

print("StoreMeanSales sample:")
print(train_df[["Store","StoreMeanSales"]].drop_duplicates().head(8).round(0))
print(f"\nMissing in train: {train_df['StoreMeanSales'].isnull().sum()}")
print(f"Missing in val:   {val_df['StoreMeanSales'].isnull().sum()}")

StoreMeanSales sample:
   Store  StoreMeanSales
0      1          4798.0
1      2          4927.0
2      3          6905.0
3      4          9574.0
4      5          4664.0
5      6          5626.0
6      7          8718.0
7      8          5374.0

Missing in train: 0
Missing in val:   0


In [36]:
FEATURES_V2 = FEATURES + ['StoreMeanSales']

lgb_t2 = lgb.Dataset(train_df[FEATURES_V2], label=y_train)
lgb_v2 = lgb.Dataset(val_df[FEATURES_V2], label=y_val, reference=lgb_t2)

model_v2 = lgb.train(
    params,
    lgb_t2,
    num_boost_round=5000,
    valid_sets=[lgb_v2],
    callbacks=callbacks
)

preds_v2 = np.maximum(model_v2.predict(val_df[FEATURES_V2]), 0)
rmspe_v2 = rmspe(y_val.values, preds_v2)

print(f"\nV1 RMSPE (no store mean):  {rmspe_v1:.4f}")
print(f"V2 RMSPE (+ store mean):    {rmspe_v2:.4f}")
print(f"Best iteration: {model_v2.best_iteration}")


[500]	valid_0's rmse: 1126.05
[1000]	valid_0's rmse: 1083.33
[1500]	valid_0's rmse: 1062.01
[2000]	valid_0's rmse: 1049.79
[2500]	valid_0's rmse: 1042.23
[3000]	valid_0's rmse: 1036.4
[3500]	valid_0's rmse: 1032.22
[4000]	valid_0's rmse: 1029.88
[4500]	valid_0's rmse: 1026.97

V1 RMSPE (no store mean):  0.1606
V2 RMSPE (+ store mean):    0.1414
Best iteration: 4540


## V3 — Add StoreDowMean

Different stores peak on different days of the week.
A Store × DayOfWeek mean captures this interaction directly.

In [37]:
dow_mean = train_df.groupby(['Store', 'DayOfWeek'])['Sales'].mean()

train_df['StoreDowMean'] = train_df.set_index(['Store', 'DayOfWeek']).index.map(dow_mean)
val_df['StoreDowMean'] = val_df.set_index(['Store', 'DayOfWeek']).index.map(dow_mean)

print(f"Missing in train: {train_df['StoreDowMean'].isnull().sum()}")
print(f"Missing in val:   {val_df['StoreDowMean'].isnull().sum()}")
print("\nSample — Store 1 by day:")
print(train_df[train_df["Store"]==1][["Store","DayOfWeek","StoreDowMean"]].drop_duplicates().sort_values("DayOfWeek").round(0))

Missing in train: 0
Missing in val:   0

Sample — Store 1 by day:
      Store  DayOfWeek  StoreDowMean
1115      1          1        5213.0
0         1          2        4715.0
5573      1          3        4582.0
4464      1          4        4469.0
3356      1          5        4805.0
2249      1          6        4988.0


In [38]:
FEATURES_V3 = FEATURES_V2 + ['StoreDowMean']

lgb_t3 = lgb.Dataset(train_df[FEATURES_V3], label=y_train)
lgb_v3 = lgb.Dataset(val_df[FEATURES_V3], label=y_val, reference=lgb_t3)

model_v3 = lgb.train(
    params,
    lgb_t3,
    num_boost_round=5000,
    valid_sets=[lgb_v3],
    callbacks=callbacks
)

preds_v3 = np.maximum(model_v3.predict(val_df[FEATURES_V3]), 0)
rmspe_v3 = rmspe(y_val.values, preds_v3)

print(f"V2 RMSPE (+ store mean):   {rmspe_v2:.4f}")
print(f"V3 RMSPE (+ StoreDowMean):    {rmspe_v3:.4f}")
print(f"Best iteration: {model_v3.best_iteration}")

[500]	valid_0's rmse: 1056.1
[1000]	valid_0's rmse: 1033.83
[1500]	valid_0's rmse: 1025.43
V2 RMSPE (+ store mean):   0.1414
V3 RMSPE (+ StoreDowMean):    0.1372
Best iteration: 1584


## StorePromoLift — tested and dropped

Tested adding per-store promo lift (difference in mean sales promo vs non-promo).
Scored 0.1372 — identical to V3. StoreMeanSales and StoreDowMean had already
captured this signal implicitly. Redundant features add noise without value.

## V4 — Add DaysToEaster

Evaluation showed April had the worst monthly error (11.9%).
Root cause: Easter is a floating holiday — falls in March some years, April others.
A fixed Month feature cannot distinguish 'April with Easter' from 'April without Easter'.

Fix: calculate how many days each row is from the nearest Easter date.
Negative = before Easter, positive = after Easter, 0 = Easter day itself.

In [39]:
easter_dates = pd.to_datetime([
    "2013-03-31",
    "2014-04-20",
    "2015-04-05"
])

def days_to_nearest_easter(row):
    date = pd.Timestamp(year=int(row["Year"]), month=int(row["Month"]), day=int(row["Day"]))
    diffs = [(date - e).days for e in easter_dates]
    return min(diffs, key=abs)

train_df["DaysToEaster"] = train_df.apply(days_to_nearest_easter, axis=1)
val_df["DaysToEaster"]   = val_df.apply(days_to_nearest_easter, axis=1)

print("DaysToEaster sample:")
print(train_df[["Year","Month","Day","DaysToEaster"]].drop_duplicates().sort_values(["Year","Month","Day"]).head(10))
print(f"\nRange: {train_df['DaysToEaster'].min()} to {train_df['DaysToEaster'].max()}")

DaysToEaster sample:
       Year  Month  Day  DaysToEaster
6684   2013      1    1           -89
5573   2013      1    2           -88
4464   2013      1    3           -87
3356   2013      1    4           -86
2249   2013      1    5           -85
2230   2013      1    6           -84
12244  2013      1    7           -83
11139  2013      1    8           -82
10034  2013      1    9           -81
8929   2013      1   10           -80

Range: -192 to 192


In [40]:
FEATURES_V4 = FEATURES_V3 + ["DaysToEaster"] 

lgb_t4 = lgb.Dataset(train_df[FEATURES_V4], label=y_train)
lgb_v4   = lgb.Dataset(val_df[FEATURES_V4], label=y_val, reference=lgb_t4)

model_v4 = lgb.train(
    params,
    lgb_t4,
    num_boost_round=5000,
    valid_sets=[lgb_v4],
    callbacks=callbacks
)

preds_v4 = np.maximum(model_v4.predict(val_df[FEATURES_V4]), 0)
rmspe_v4 = rmspe(y_val.values, preds_v4)

print(f"V3 RMSPE (+ StoreDowMean):  0.1372")
print(f"V4 RMSPE (+ DaysToEaster):  {rmspe_v4:.4f}")
print(f"Best iteration: {model_v4.best_iteration}")

[500]	valid_0's rmse: 974.173
[1000]	valid_0's rmse: 940.458
[1500]	valid_0's rmse: 920.557
[2000]	valid_0's rmse: 908.767
[2500]	valid_0's rmse: 901.838
[3000]	valid_0's rmse: 895.703
[3500]	valid_0's rmse: 891.932
[4000]	valid_0's rmse: 889.067
[4500]	valid_0's rmse: 886.147
V3 RMSPE (+ StoreDowMean):  0.1372
V4 RMSPE (+ DaysToEaster):  0.1163
Best iteration: 4605


## V5 — Log-transform the target

Sales has a right skew — some stores reach £40,000 while most are £4,000–8,000.
LightGBM trains on RMSE internally, which penalises large absolute errors.
High-value stores dominate the loss and pull attention from typical stores.

Predicting log(Sales) compresses the skew so all stores contribute equally.
np.log1p avoids log(0) issues. np.expm1 reverses the transform after prediction.

In [47]:
train_df['LogSales'] = np.log1p(train_df['Sales'])
val_df['LogSales'] = np.log1p(val_df['Sales'])

LOG_TARGET = "LogSales"
FEATURES_V5 = FEATURES_V4.copy()
params_v5    = params.copy()

lgb_t5 = lgb.Dataset(train_df[FEATURES_V5], label=train_df[LOG_TARGET])
lgb_v5   = lgb.Dataset(val_df[FEATURES_V5], label=val_df[LOG_TARGET], reference=lgb_t5)

model_v5 = lgb.train(
    params_v5,
    lgb_t5,
    num_boost_round=5000,
    valid_sets=[lgb_v5],
    callbacks=callbacks
)

log_preds = model_v5.predict(val_df[FEATURES_V5])
preds_v5 = np.maximum(np.expm1(log_preds), 0)

rmspe_v5 = rmspe(val_df["Sales"].values, preds_v5)

print(f"\nV4 RMSPE (no log transform): {rmspe_v4:.4f}")
print(f"V5 RMSPE (log transform): {rmspe_v5:.4f}")
print(f"Best iteration: {model_v5.best_iteration}")

[500]	valid_0's rmse: 0.127594
[1000]	valid_0's rmse: 0.123414
[1500]	valid_0's rmse: 0.120477
[2000]	valid_0's rmse: 0.118762
[2500]	valid_0's rmse: 0.11769
[3000]	valid_0's rmse: 0.116896
[3500]	valid_0's rmse: 0.116363

V4 RMSPE (no log transform): 0.1163
V5 RMSPE (log transform): 0.1146
Best iteration: 3912


## V6 — Lower learning rate + more leaves

Drop learning rate from 0.05 to 0.02 for more careful learning.
Increase num_leaves from 63 to 127 — larger dataset with more features
justifies more complex trees.

In [48]:
params_v6 = params_v5.copy()
params_v6["learning_rate"] = 0.02
params_v6["num_leaves"] = 127

callbacks_v6 = [
    lgb.early_stopping(stopping_rounds=100, verbose=False),
    lgb.log_evaluation(period=500)
]

lgb_t6 = lgb.Dataset(train_df[FEATURES_V5], label=train_df[LOG_TARGET])
lgb_v6   = lgb.Dataset(val_df[FEATURES_V5], label=val_df[LOG_TARGET], reference=lgb_t6)


model_v6 = lgb.train(
    params_v6,
    lgb_t6,
    num_boost_round=10000,
    valid_sets=[lgb_v6],
    callbacks=callbacks_v6
)

log_preds_v6 = model_v6.predict(val_df[FEATURES_V5])
preds_v6 = np.maximum(np.expm1(log_preds_v6), 0)
rmspe_v6     = rmspe(val_df["Sales"].values, preds_v6)

print(f"\nV5 RMSPE (lr=0.05, leaves=63):   {rmspe_v5:.4f}")
print(f"V6 RMSPE (lr=0.02, leaves=127): {rmspe_v6:.4f}")
print(f"Best iteration: {model_v6.best_iteration}")

[500]	valid_0's rmse: 0.131448
[1000]	valid_0's rmse: 0.12637
[1500]	valid_0's rmse: 0.123781
[2000]	valid_0's rmse: 0.121983
[2500]	valid_0's rmse: 0.120601
[3000]	valid_0's rmse: 0.119564
[3500]	valid_0's rmse: 0.118744
[4000]	valid_0's rmse: 0.118148
[4500]	valid_0's rmse: 0.117636
[5000]	valid_0's rmse: 0.117282
[5500]	valid_0's rmse: 0.116935
[6000]	valid_0's rmse: 0.116683

V5 RMSPE (lr=0.05, leaves=63):   0.1146
V6 RMSPE (lr=0.02, leaves=127): 0.1149
Best iteration: 6156


## V7 — Final tuning (lr=0.01, min_child_samples=40)

Halve learning rate again for final polish.
min_child_samples=50 was tested and over-regularised (scored 0.1311).
Settled on 40 as the best value through testing.

In [43]:
params_v7 = params_v6.copy()
params_v7["learning_rate"] = 0.01
params_v7["min_child_samples"] = 40

callbacks_v7 = [
    lgb.early_stopping(stopping_rounds=150, verbose=False),
    lgb.log_evaluation(period=1000)
]

lgb_t7 = lgb.Dataset(train_df[FEATURES_V5], label=train_df[LOG_TARGET])
lgb_v7 = lgb.Dataset(val_df[FEATURES_V5],   label=val_df[LOG_TARGET], reference=lgb_t7)

model_v7 = lgb.train(
    params_v7,
    lgb_t7,
    num_boost_round=20000,
    valid_sets=[lgb_v7],
    callbacks=callbacks_v7
)

log_preds_v7 = model_v7.predict(val_df[FEATURES_V5])
preds_v7 = np.maximum(np.expm1(log_preds_v7), 0)
rmspe_v7 = rmspe(val_df["Sales"].values, preds_v7)

print(f"\nV6 RMSPE: {rmspe_v6:.4f}")
print(f"V7 RMSPE: {rmspe_v7:.4f}")
print(f"Best iteration: {model_v7.best_iteration}")

[1000]	valid_0's rmse: 0.131477
[2000]	valid_0's rmse: 0.126009
[3000]	valid_0's rmse: 0.123341
[4000]	valid_0's rmse: 0.12163
[5000]	valid_0's rmse: 0.120178
[6000]	valid_0's rmse: 0.119152
[7000]	valid_0's rmse: 0.118311
[8000]	valid_0's rmse: 0.117673
[9000]	valid_0's rmse: 0.117104
[10000]	valid_0's rmse: 0.116719
[11000]	valid_0's rmse: 0.116419
[12000]	valid_0's rmse: 0.116179
[13000]	valid_0's rmse: 0.115976
[14000]	valid_0's rmse: 0.115782

V6 RMSPE: 0.1149
V7 RMSPE: 0.1140
Best iteration: 14309


In [49]:
os.makedirs('../models', exist_ok=True)
model_v7.save_model('../models/lgb_model_v7.txt')
print('Model saved to ../models/lgb_model_v7.txt')

Model saved to ../models/lgb_model_v7.txt


## Improvement Summary

| Version | Change | RMSPE | Delta |
|---------|--------|-------|-------|
| Original | 998 rounds, base features | 0.1852 | — |
| V1 | Fixed rounds — converged at 3581 | 0.1606 | −0.025 |
| V2 | + StoreMeanSales | 0.1414 | −0.019 |
| V3 | + StoreDowMean | 0.1372 | −0.004 |
| StorePromoLift | Tested — no gain, dropped | 0.1372 | 0.000 |
| V4 | + DaysToEaster | 0.1162 | −0.021 |
| V5 | Log-transform target | 0.1146 | −0.002 |
| V6 | lr=0.02, num_leaves=127 | 0.1149 | — |
| **V7** | **lr=0.01, min_child_samples=40** | **0.1140** | **−0.071 total** |

### Key lessons
- Fixing an unconverged model gave the first gain at zero feature cost
- Per-store mean features (V2, V3) provided the largest single improvements
- DaysToEaster was the insight gain — domain knowledge beat any tuning effort
- StorePromoLift added nothing — already captured by StoreMeanSales
- min_child_samples=50 over-regularised — always test, don't assume
- Optuna (08_tuning.ipynb) ran 50 trials and could not beat the manual V7 config
- Total improvement: 0.1852 → 0.1140 — 38% reduction in RMSPE